In [8]:
import osmium



In [30]:

LINK_TYPES = {
    "motorway_link",
    "trunk_link",
    "primary_link",
    "secondary_link",
    "tertiary_link",
}

class NodeHandler(osmium.SimpleHandler):
    
    def node(self, n):
        self.nodes.append({'id': n.id, 'location': n.location, 'tags': n.tags})
        
    def __init__(self):
        super().__init__()
        self.nodes = []
        self.links = []

    def way(self, w):
        highway = w.tags.get("highway")
        if highway in LINK_TYPES:
            self.links.append({
                "id": w.id,
                "type": highway,
                "nodes": [n.ref for n in w.nodes],
                "tags": dict(w.tags),
            })
    


In [31]:
handler = NodeHandler()
handler.apply_file("../data/osm-pbf/048_Firenze-2026-04-30T02Z.osm.pbf", locations=True)

In [32]:
len(handler.nodes), len(handler.links)

(3804506, 876)

In [33]:
[handler.nodes[i] for i in range(10)]

[{'id': 688348,
  'location': osmium.osm.Location(x=111870201, y=438866134),
  'tags': osmium.osm.TagList({<invalid>})},
 {'id': 688349,
  'location': osmium.osm.Location(x=111860941, y=438859728),
  'tags': osmium.osm.TagList({<invalid>})},
 {'id': 688350,
  'location': osmium.osm.Location(x=111855794, y=438852873),
  'tags': osmium.osm.TagList({<invalid>})},
 {'id': 688351,
  'location': osmium.osm.Location(x=111845000, y=438846109),
  'tags': osmium.osm.TagList({<invalid>})},
 {'id': 688352,
  'location': osmium.osm.Location(x=111819475, y=438836644),
  'tags': osmium.osm.TagList({<invalid>})},
 {'id': 688354,
  'location': osmium.osm.Location(x=111813459, y=438835305),
  'tags': osmium.osm.TagList({<invalid>})},
 {'id': 688366,
  'location': osmium.osm.Location(x=111726135, y=438737425),
  'tags': osmium.osm.TagList({<invalid>})},
 {'id': 806372,
  'location': osmium.osm.Location(x=111661140, y=438652715),
  'tags': osmium.osm.TagList({<invalid>})},
 {'id': 806426,
  'location': os

In [34]:
[handler.links[i] for i in range(10)]

[{'id': 4621987,
  'type': 'motorway_link',
  'nodes': [29348385,
   1913940364,
   8846881818,
   8846881819,
   29348387,
   29348389,
   8846881817,
   292623147,
   29348392,
   29352877,
   8846881816,
   8846881815,
   29352878,
   8846881813,
   8846881814,
   29352875],
  'tags': {'highway': 'motorway_link',
   'lane_markings': 'no',
   'lanes': '1',
   'long_name': 'Svincolo di San Casciano Val di Pesa',
   'maxspeed': '40',
   'name': 'Svincolo di San Casciano',
   'oneway': 'yes',
   'surface': 'asphalt'}},
 {'id': 4622639,
  'type': 'motorway_link',
  'nodes': [4048421,
   8922873432,
   8922873433,
   29359387,
   8846881803,
   8846881802,
   8846881801,
   29359386,
   8846881800,
   29359385,
   8846881799,
   29359383,
   8846881798,
   29359382,
   8846881797,
   4416204870,
   29359381],
  'tags': {'highway': 'motorway_link',
   'lane_markings': 'no',
   'long_name': 'Svincolo di San Casciano Val di Pesa',
   'maxspeed': '40',
   'name': 'Svincolo di San Casciano',
 

---

In [3]:
import osmium
import osmium.geom
import math
from xml.etree.ElementTree import Element, SubElement, ElementTree, indent
import pyproj

# ── Projection: WGS84 → UTM (change EPSG to match your region) ──────────────
# EPSG:32632 = UTM zone 32N (central Italy / central Europe)
# Adjust to your area: https://epsg.io
PROJECTION = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:32632", always_xy=True)

# ── Highway types to include, with default speed (m/s) and capacity ──────────
HIGHWAY_DEFAULTS = {
    "motorway":       {"freespeed": 33.333, "capacity": 2000.0, "lanes": 2},
    "motorway_link":  {"freespeed": 22.222, "capacity": 1500.0, "lanes": 1},
    "trunk":          {"freespeed": 22.222, "capacity": 2000.0, "lanes": 2},
    "trunk_link":     {"freespeed": 13.889, "capacity": 1500.0, "lanes": 1},
    "primary":        {"freespeed": 22.222, "capacity": 1500.0, "lanes": 1},
    "primary_link":   {"freespeed": 13.889, "capacity": 1500.0, "lanes": 1},
    "secondary":      {"freespeed": 13.889, "capacity": 1000.0, "lanes": 1},
    "secondary_link": {"freespeed": 8.333,  "capacity": 800.0,  "lanes": 1},
    "tertiary":       {"freespeed": 8.333,  "capacity": 600.0,  "lanes": 1},
    "tertiary_link":  {"freespeed": 8.333,  "capacity": 600.0,  "lanes": 1},
    "residential":    {"freespeed": 8.333,  "capacity": 300.0,  "lanes": 1},
    "living_street":  {"freespeed": 2.778,  "capacity": 100.0,  "lanes": 1},
    "unclassified":   {"freespeed": 8.333,  "capacity": 300.0,  "lanes": 1},
}

NODE_TYPE = "90"  # default node type attribute


def haversine_m(lon1, lat1, lon2, lat2):
    """Great-circle distance in metres between two WGS84 points."""
    R = 6_371_000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


def project(lon, lat):
    """Return projected (x, y) in metres, rounded to 1 decimal."""
    x, y = PROJECTION.transform(lon, lat)
    return round(x, 1), round(y, 1)


# ── Pass 1: collect all node IDs referenced by relevant ways ─────────────────

class WayScanner(osmium.SimpleHandler):
    """First pass: find which OSM node IDs are used by relevant ways."""

    def __init__(self):
        super().__init__()
        self.needed_nodes: set[int] = set()
        self.ways: list[dict] = []

    def way(self, w):
        hw = w.tags.get("highway")
        if hw not in HIGHWAY_DEFAULTS:
            return
        node_refs = [n.ref for n in w.nodes]
        if len(node_refs) < 2:
            return
        self.needed_nodes.update(node_refs)
        self.ways.append({
            "id":      w.id,
            "nodes":   node_refs,
            "highway": hw,
            "oneway":  w.tags.get("oneway", "no"),
            "lanes":   w.tags.get("lanes"),
            "maxspeed": w.tags.get("maxspeed"),
        })


# ── Pass 2: collect coordinates for needed nodes ──────────────────────────────

class NodeCollector(osmium.SimpleHandler):
    """Second pass: store (lon, lat) for every needed node ID."""

    def __init__(self, needed: set[int]):
        super().__init__()
        self.needed = needed
        self.coords: dict[int, tuple[float, float]] = {}

    def node(self, n):
        if n.id in self.needed:
            self.coords[n.id] = (n.location.lon, n.location.lat)


# ── Build XML ─────────────────────────────────────────────────────────────────

def parse_maxspeed(tag_value: str | None) -> float | None:
    """Parse maxspeed tag ('50', '50 mph', '30 kmh') → m/s, or None."""
    if not tag_value:
        return None
    tag_value = tag_value.strip().lower()
    try:
        if "mph" in tag_value:
            return float(tag_value.replace("mph", "").strip()) * 0.44704
        kmh = float(tag_value.replace("km/h", "").replace("kmh", "").strip())
        return kmh / 3.6
    except ValueError:
        return None


def build_xml(pbf_path: str, output_path: str, cap_period: str = "01:00:00") -> None:
    print(f"[1/3] Scanning ways in {pbf_path} ...")
    scanner = WayScanner()
    scanner.apply_file(pbf_path)
    print(f"      → {len(scanner.ways)} relevant ways, {len(scanner.needed_nodes)} unique nodes")

    print("[2/3] Collecting node coordinates ...")
    collector = NodeCollector(scanner.needed_nodes)
    collector.apply_file(pbf_path)
    print(f"      → resolved {len(collector.coords)} node coordinates")

    print("[3/3] Building XML ...")
    root = Element("network")

    # ── <nodes> ───────────────────────────────────────────────────────────────
    nodes_el = SubElement(root, "nodes")

    # Only emit nodes that were actually resolved
    resolved = collector.coords
    used_node_ids: set[int] = set()
    for way in scanner.ways:
        for nid in way["nodes"]:
            if nid in resolved:
                used_node_ids.add(nid)

    for nid in sorted(used_node_ids):
        lon, lat = resolved[nid]
        x, y = project(lon, lat)
        SubElement(nodes_el, "node", {
            "id":   str(nid),
            "x":    str(x),
            "y":    str(y),
            "type": NODE_TYPE,
        })

    # ── <links> ───────────────────────────────────────────────────────────────
    links_el = SubElement(root, "links", {"capperiod": cap_period})
    link_id = 1

    for way in scanner.ways:
        hw = way["highway"]
        defaults = HIGHWAY_DEFAULTS[hw]

        # Speed
        freespeed = parse_maxspeed(way["maxspeed"]) or defaults["freespeed"]

        # Lanes
        try:
            lanes = int(way["lanes"]) if way["lanes"] else defaults["lanes"]
        except ValueError:
            lanes = defaults["lanes"]

        capacity = defaults["capacity"] * lanes

        # Oneway logic
        oneway_tag = way["oneway"]
        is_oneway = oneway_tag in ("yes", "1", "true") or hw in ("motorway", "motorway_link")
        is_reverse = oneway_tag == "-1"

        node_refs = way["nodes"]

        # Walk consecutive node pairs to create individual link segments
        for i in range(len(node_refs) - 1):
            from_id = node_refs[i]
            to_id   = node_refs[i + 1]

            if from_id not in resolved or to_id not in resolved:
                continue  # skip if coordinates missing (border nodes)

            flon, flat = resolved[from_id]
            tlon, tlat = resolved[to_id]
            length = round(haversine_m(flon, flat, tlon, tlat), 1)

            base_attrs = {
                "length":    str(length),
                "freespeed": str(round(freespeed, 4)),
                "capacity":  str(capacity),
                "permlanes": str(lanes),
                "origid":    str(way["id"]),
            }

            if not is_reverse:
                SubElement(links_el, "link", {
                    "id":     str(link_id),
                    "from":   str(from_id),
                    "to":     str(to_id),
                    "oneway": "1",
                    **base_attrs,
                })
                link_id += 1

            if not is_oneway:  # add reverse link
                SubElement(links_el, "link", {
                    "id":     str(link_id),
                    "from":   str(to_id),
                    "to":     str(from_id),
                    "oneway": "1",
                    **base_attrs,
                })
                link_id += 1

            if is_reverse:  # oneway="-1": only reverse direction
                SubElement(links_el, "link", {
                    "id":     str(link_id),
                    "from":   str(to_id),
                    "to":     str(from_id),
                    "oneway": "1",
                    **base_attrs,
                })
                link_id += 1

    # ── Write file ────────────────────────────────────────────────────────────
    indent(root, space="\t")
    tree = ElementTree(root)
    with open(output_path, "wb") as f:
        f.write(b'<?xml version="1.0" encoding="UTF-8"?>\n')
        tree.write(f, encoding="unicode" if False else "utf-8", xml_declaration=False)

    print(f"\nDone! Written to {output_path}")
    print(f"  Nodes : {len(used_node_ids)}")
    print(f"  Links : {link_id - 1}")


In [4]:
build_xml(
        pbf_path="../data/osm-pbf/048_Firenze-2026-04-30T02Z.osm.pbf",
        output_path="network.xml",
        cap_period="01:00:00",
    )

[1/3] Scanning ways in ../data/osm-pbf/048_Firenze-2026-04-30T02Z.osm.pbf ...
      → 36482 relevant ways, 326666 unique nodes
[2/3] Collecting node coordinates ...
      → resolved 326666 node coordinates
[3/3] Building XML ...

Done! Written to network.xml
  Nodes : 326666
  Links : 597762
